# CE 477 – Data Science: Assignment #2 (Part 1: Classification)

**Institution:** Izmir University of Economics  
**Department:** Department of Computer Engineering  
**Dataset:** AFAD Seismic Activity Catalog (Preprocessed Dataset, $N = 153,439$)

---

## 1. Problem Formulation and Target Attribute Selection

In accordance with **Assignment #2 – Task 1 (Classification)**, this notebook implements an end-to-end supervised classification pipeline on the preprocessed AFAD earthquake catalog (`data/afad_earthquake_data_preprocessed.csv`):

1. **Discrete Target Attribute (`target_class`):** We utilize the discretized earthquake severity attribute derived from continuous magnitude measurements (`magnitude_class`), consolidated into three seismologically meaningful and statistically well-supported severity tiers:
   - **Class 0 – `Micro (<3.0)`:** Background micro-seismicity ($102,656$ events, $66.90\%$).
   - **Class 1 – `Minor (3.0-3.9)`:** Felt local minor earthquakes ($45,983$ events, $29.97\%$).
   - **Class 2 – `Light+ (>=4.0)`:** Significant and potentially damaging seismic events combining Light ($4.0$–$4.9$), Moderate ($5.0$–$5.9$), and Strong+ ($\ge 6.0$) shocks ($4,800$ events, $3.13\%$).
2. **Strict Data Leakage Prevention:** All direct magnitude representations (`magnitude`, `magnitude_norm`, `magnitude_std`, `is_major`) and global full-dataset projections (`PCA1`, `PCA2`, `PCA3`) are strictly excluded from the predictor matrix $X$.
3. **Data Partitioning:** Stratified **60% Training** ($92,063$ rows), **20% Validation** ($30,688$ rows), and **20% Testing** ($30,688$ rows) splits ensure identical class proportions across all partitions while preventing feature engineering or scaling leakage.
4. **Classification Algorithms Evaluated:**
   - **Decision Tree Classifier** *(Mandatory white-box hierarchical rule learner)*
   - **Random Forest Classifier** *(Bagging tree ensemble)*
   - **Logistic Regression** *(Multinomial linear probabilistic baseline)*
   - **Gradient Boosting Classifier** *(Histogram-based boosted decision trees)*

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
os.makedirs("../figures", exist_ok=True)
print("Environment initialized successfully.")

Environment initialized successfully.


### Analysis of Environment and Tooling
The core scientific computing stack (`numpy`, `pandas`, `scikit-learn`, `matplotlib`, and `seaborn`) is configured with deterministic random seeds (`random_state=42`) to guarantee full reproducibility across training, validation, and testing splits.

In [2]:
# Load preprocessed AFAD earthquake catalog
df = pd.read_csv("../data/afad_earthquake_data_preprocessed.csv")
df = df.loc[:, ~df.columns.duplicated()]
print(f"Loaded dataset shape: {df.shape}")

# Parse datetime for cyclical temporal feature extraction
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df["month"] = df["datetime"].dt.month.fillna(6).astype(int)
df["hour"] = df["datetime"].dt.hour.fillna(12).astype(int)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12.0)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12.0)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)

# Map discretized magnitude_class into 3 ordinal severity classes
class_mapping = {
    "Micro (<3.0)": 0,
    "Minor (3.0-3.9)": 1,
    "Light (4.0-4.9)": 2,
    "Moderate (5.0-5.9)": 2,
    "Strong+ (>=6.0)": 2,
}
class_names = ["Micro (<3.0)", "Minor (3.0-3.9)", "Light+ (>=4.0)"]
df["target_class"] = df["magnitude_class"].map(class_mapping)
print(f"Missing target values after mapping: {df['target_class'].isna().sum()}\n")

print("=== Target Attribute Distribution (Discretized Magnitude Severity) ===")
for idx, name in enumerate(class_names):
    cnt = int((df["target_class"] == idx).sum())
    pct = 100.0 * cnt / len(df)
    print(f"  Class {idx} | {name:<15} : {cnt:>8,} events ({pct:>5.2f}%)")

Loaded dataset shape: (153439, 36)
Missing target values after mapping: 0

=== Target Attribute Distribution (Discretized Magnitude Severity) ===
  Class 0 | Micro (<3.0)    :  102,656 events (66.90%)
  Class 1 | Minor (3.0-3.9) :   45,983 events (29.97%)
  Class 2 | Light+ (>=4.0)  :    4,800 events ( 3.13%)


### Analysis of Target Attribute Distribution
The target attribute exhibits the power-law frequency distribution governed by the **Gutenberg–Richter law** ($\log_{10} N = a - bM$), where earthquake frequency decays exponentially with increasing magnitude:
- **Micro earthquakes ($<3.0$)** account for **$66.90\%$** ($102,656$ records) of the catalog.
- **Minor earthquakes ($3.0$–$3.9$)** constitute **$29.97\%$** ($45,983$ records).
- **Light and stronger events ($\ge 4.0$)** represent **$3.13\%$** ($4,800$ records).

Consolidating events with $M \ge 4.0$ into `Light+ (>=4.0)` yields $960$ test samples and $960$ validation samples in the 20% partitions, ensuring statistically low-variance estimates of Precision, Recall, and F1-score while preserving the natural multi-class imbalance challenge of real-world seismic catalogs.

In [3]:
# Select strictly non-leaking spatial, temporal, and instrumentation features
base_features = [
    "latitude", "longitude", "depth", "rms", "year",
    "month_sin", "month_cos", "hour_sin", "hour_cos",
    "type_ML", "type_Md", "type_Ml", "type_Mw", "type_Other",
    "province"
]

X_raw = df[base_features].copy()
y = df["target_class"].astype(int).values

# Step 1: 60% Training vs 40% Temporary split (Stratified)
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X_raw, y, test_size=0.40, stratify=y, random_state=42
)

# Step 2: Split 40% Temporary equally into 20% Validation and 20% Testing (Stratified)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

# Step 3: Fit spatial province statistics strictly on the 60% Training set
prov_freq = X_train_raw["province"].value_counts(normalize=True).to_dict()
train_tmp = X_train_raw.copy()
train_tmp["target"] = y_train
prov_target_rate = train_tmp.groupby("province")["target"].mean().to_dict()
global_target_mean = float(np.mean(y_train))

def engineer_features(part_df):
    out = part_df.drop(columns=["province"]).copy()
    out["province_freq"] = part_df["province"].map(prov_freq).fillna(0.0)
    out["province_seismic_index"] = part_df["province"].map(prov_target_rate).fillna(global_target_mean)
    out["log_depth"] = np.log1p(np.maximum(0.0, out["depth"]))
    out["log_rms"] = np.log1p(np.maximum(0.0, out["rms"]))
    out["lat_lon_interaction"] = out["latitude"] * out["longitude"]
    return out

X_train_fe = engineer_features(X_train_raw)
X_val_fe = engineer_features(X_val_raw)
X_test_fe = engineer_features(X_test_raw)
feature_names = list(X_train_fe.columns)

# Step 4: Fit StandardScaler strictly on Training continuous attributes
continuous_cols = [
    "latitude", "longitude", "depth", "rms", "year",
    "month_sin", "month_cos", "hour_sin", "hour_cos",
    "province_freq", "province_seismic_index",
    "log_depth", "log_rms", "lat_lon_interaction"
]
scaler = StandardScaler()
X_train_scaled = X_train_fe.copy()
X_val_scaled = X_val_fe.copy()
X_test_scaled = X_test_fe.copy()

X_train_scaled[continuous_cols] = scaler.fit_transform(X_train_fe[continuous_cols])
X_val_scaled[continuous_cols] = scaler.transform(X_val_fe[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test_fe[continuous_cols])

print("=== Stratified 60% / 20% / 20% Data Split Summary ===")
print(f"  Training Set   (60%): {len(X_train_fe):,} rows | Class counts: {np.bincount(y_train).tolist()}")
print(f"  Validation Set (20%): {len(X_val_fe):,} rows | Class counts: {np.bincount(y_val).tolist()}")
print(f"  Testing Set    (20%): {len(X_test_fe):,} rows | Class counts: {np.bincount(y_test).tolist()}\n")
print(f"Total leakage-free predictor features ({len(feature_names)}):\n  {feature_names}")

=== Stratified 60% / 20% / 20% Data Split Summary ===
  Training Set   (60%): 92,063 rows | Class counts: [61593, 27590, 2880]
  Validation Set (20%): 30,688 rows | Class counts: [20531, 9197, 960]
  Testing Set    (20%): 30,688 rows | Class counts: [20532, 9196, 960]

Total leakage-free predictor features (19):
  ['latitude', 'longitude', 'depth', 'rms', 'year', 'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'type_ML', 'type_Md', 'type_Ml', 'type_Mw', 'type_Other', 'province_freq', 'province_seismic_index', 'log_depth', 'log_rms', 'lat_lon_interaction']


### Analysis of Stratified Partitioning and Feature Engineering
1. **Exact 60% / 20% / 20% Proportions:** Each split preserves the exact class distribution ($66.90\%$ `Micro`, $29.97\%$ `Minor`, and $3.13\%$ `Light+`).
2. **Strict Separation of Training Statistics:** Both regional encoders (`province_freq` and `province_seismic_index`) and `StandardScaler` parameters ($\mu, \sigma$) are estimated exclusively on the $92,063$ training observations and subsequently applied to the validation and testing sets. This eliminates any information leakage from validation or test samples into model training.
3. **Domain-Specific Non-Linear Features:** Logarithmic transforms (`log_depth`, `log_rms`) compress right-skewed hypocenter depth and waveform root-mean-square travel-time residual distributions, helping linear models (`LogisticRegression`) as well as tree splits.
4. **Known limitation (target encoding):** `province_seismic_index` is a mean-target encoding estimated on the training split and then also applied to the training rows themselves (in-sample). Validation and testing rows never contribute to it, but a 5-fold out-of-fold variant is evaluated as a sensitivity check in Section 5.8.

In [4]:
# Hyperparameter tuning for Decision Tree on the 20% Validation Set
depths_to_try = [4, 6, 8, 10, 12, 14, 16, 20]
dt_tuning_records = []
best_dt_depth = None
best_dt_val_f1 = -1.0

print("=== Decision Tree Validation Set Hyperparameter Tuning ===")
print("  max_depth | Train Weighted F1 | Val Accuracy | Val Weighted F1 | Val Macro F1")
print("  -----------------------------------------------------------------------------")

for d in depths_to_try:
    dt_cand = DecisionTreeClassifier(max_depth=d, min_samples_leaf=25, criterion="gini", random_state=42)
    dt_cand.fit(X_train_fe, y_train)
    tr_pred = dt_cand.predict(X_train_fe)
    va_pred = dt_cand.predict(X_val_fe)
    tr_wf1 = f1_score(y_train, tr_pred, average="weighted")
    va_acc = accuracy_score(y_val, va_pred)
    va_wf1 = f1_score(y_val, va_pred, average="weighted")
    va_mf1 = f1_score(y_val, va_pred, average="macro")
    dt_tuning_records.append({
        "max_depth": d,
        "train_weighted_f1": tr_wf1,
        "val_accuracy": va_acc,
        "val_weighted_f1": va_wf1,
        "val_macro_f1": va_mf1,
    })
    print(f"  {d:>9} | {tr_wf1:>17.4f} | {va_acc:>12.4f} | {va_wf1:>15.4f} | {va_mf1:>12.4f}")
    if va_wf1 > best_dt_val_f1:
        best_dt_val_f1 = va_wf1
        best_dt_depth = d

print(f"\nSelected Optimal Decision Tree Depth: max_depth = {best_dt_depth} (Validation Weighted F1 = {best_dt_val_f1:.4f})")

# Train all 4 classification algorithms on the 60% Training Set
models = {
    "Decision Tree": (
        DecisionTreeClassifier(max_depth=best_dt_depth, min_samples_leaf=25, criterion="gini", random_state=42),
        False
    ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=150, max_depth=16, min_samples_leaf=10, n_jobs=-1, random_state=42),
        False
    ),
    "Logistic Regression": (
        LogisticRegression(max_iter=500, C=1.0, random_state=42),
        True
    ),
    "Gradient Boosting": (
        HistGradientBoostingClassifier(max_iter=200, max_depth=10, learning_rate=0.08, min_samples_leaf=25, random_state=42),
        False
    ),
}

trained_models = {}
val_predictions = {}
test_predictions = {}

for name, (clf, use_scaled) in models.items():
    X_tr = X_train_scaled if use_scaled else X_train_fe
    X_va = X_val_scaled if use_scaled else X_val_fe
    X_te = X_test_scaled if use_scaled else X_test_fe
    clf.fit(X_tr, y_train)
    trained_models[name] = clf
    val_predictions[name] = clf.predict(X_va)
    test_predictions[name] = clf.predict(X_te)

print("\nAll 4 classification models trained successfully on the 60% Training Set.")

=== Decision Tree Validation Set Hyperparameter Tuning ===
  max_depth | Train Weighted F1 | Val Accuracy | Val Weighted F1 | Val Macro F1
  -----------------------------------------------------------------------------


          4 |            0.6050 |       0.6969 |          0.6009 |       0.4063


          6 |            0.6896 |       0.7078 |          0.6863 |       0.5297


          8 |            0.7010 |       0.7172 |          0.6939 |       0.5262


         10 |            0.7142 |       0.7266 |          0.6995 |       0.5393


         12 |            0.7269 |       0.7302 |          0.7034 |       0.5471


         14 |            0.7397 |       0.7275 |          0.7065 |       0.5524


         16 |            0.7483 |       0.7268 |          0.7062 |       0.5540


         20 |            0.7629 |       0.7211 |          0.7071 |       0.5573

Selected Optimal Decision Tree Depth: max_depth = 20 (Validation Weighted F1 = 0.7071)



All 4 classification models trained successfully on the 60% Training Set.


### Analysis of Validation Set Hyperparameter Tuning
The validation curve demonstrates the classic bias–variance trade-off for the **Decision Tree Classifier**:
- At shallow depths (`max_depth=4`), the tree suffers from high bias (underfitting), achieving a Validation Weighted F1 of only **$0.6009$** and Macro F1 of **$0.4063$**.
- Increasing `max_depth` from $6$ to $14$ produces substantial gains in both Weighted F1 ($0.6863 \rightarrow 0.7065$) and Macro F1 ($0.5297 \rightarrow 0.5524$) as the tree captures hierarchical interactions among magnitude scale type (`type_Mw`, `type_ML`), hypocenter `depth`, travel-time residual `rms`, and tectonic coordinates (`latitude`, `longitude`).
- Beyond depth $14$ the validation curve is essentially flat: Validation Weighted F1 is $0.7065$ (depth $14$), $0.7062$ (depth $16$) and $0.7071$ (depth $20$), i.e. differences below $0.001$, while the Train-Validation gap widens from $0.033$ (depth $14$: $0.7397$ vs. $0.7065$) to $0.056$ (depth $20$: $0.7629$ vs. $0.7071$) and Validation Accuracy peaks earlier, at depth $12$ ($0.7302$). Selecting the depth with the highest validation weighted F1 gives `max_depth=20`, but depths $12$-$14$ would be an equally defensible and less over-fitted choice: the selected value reflects a plateau rather than a sharp optimum.

In [5]:
# Visualize the learned Decision Tree model (top 3 levels) alongside the validation tuning curve
dt_model = trained_models["Decision Tree"]
print("=== Learned Decision Tree Top-3 Hierarchy Rules ===")
print(export_text(dt_model, feature_names=feature_names, max_depth=3))

fig = plt.figure(figsize=(20, 10))
gs = fig.add_gridspec(1, 2, width_ratios=[2.3, 1.0], wspace=0.18)

ax_tree = fig.add_subplot(gs[0])
plot_tree(
    dt_model,
    max_depth=3,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    proportion=True,
    fontsize=8,
    precision=2,
    ax=ax_tree,
)
ax_tree.set_title(
    f"Learned Decision Tree Classifier (Top 3 Splits Shown; Optimal Tuned Depth = {best_dt_depth})",
    fontsize=13,
    fontweight="bold",
    pad=12,
)

ax_tune = fig.add_subplot(gs[1])
tune_df = pd.DataFrame(dt_tuning_records)
ax_tune.plot(tune_df["max_depth"], tune_df["train_weighted_f1"], marker="o", linestyle="--", color="gray", label="Train Weighted F1")
ax_tune.plot(tune_df["max_depth"], tune_df["val_weighted_f1"], marker="s", linewidth=2.2, color="crimson", label="Validation Weighted F1")
ax_tune.plot(tune_df["max_depth"], tune_df["val_accuracy"], marker="^", linewidth=2.0, color="navy", label="Validation Accuracy")
ax_tune.axvline(best_dt_depth, color="darkgreen", linestyle=":", linewidth=1.8, label=f"Selected Depth ({best_dt_depth})")
ax_tune.set_title("Decision Tree Hyperparameter Tuning (Validation Set)", fontsize=12, fontweight="bold")
ax_tune.set_xlabel("Maximum Tree Depth (max_depth)", fontsize=10)
ax_tune.set_ylabel("Score", fontsize=10)
ax_tune.legend(loc="lower right", fontsize=9)
ax_tune.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig("../figures/fig14_decision_tree_visualization.png", bbox_inches="tight")
plt.show()

=== Learned Decision Tree Top-3 Hierarchy Rules ===
|--- type_Mw <= 0.50
|   |--- depth <= 7.88
|   |   |--- type_Other <= 0.50
|   |   |   |--- depth <= 6.43
|   |   |   |   |--- truncated branch of depth 17
|   |   |   |--- depth >  6.43
|   |   |   |   |--- truncated branch of depth 17
|   |   |--- type_Other >  0.50
|   |   |   |--- longitude <= 26.14
|   |   |   |   |--- truncated branch of depth 4
|   |   |   |--- longitude >  26.14
|   |   |   |   |--- truncated branch of depth 9
|   |--- depth >  7.88
|   |   |--- type_Other <= 0.50
|   |   |   |--- type_ML <= 0.50
|   |   |   |   |--- truncated branch of depth 17
|   |   |   |--- type_ML >  0.50
|   |   |   |   |--- truncated branch of depth 17
|   |   |--- type_Other >  0.50
|   |   |   |--- hour_sin <= 0.79
|   |   |   |   |--- truncated branch of depth 8
|   |   |   |--- hour_sin >  0.79
|   |   |   |   |--- truncated branch of depth 4
|--- type_Mw >  0.50
|   |--- rms <= 0.48
|   |   |--- log_depth <= 2.15
|   |   |   |---

### Analysis of the Learned Decision Tree Structure (`fig14_decision_tree_visualization.png`)
Inspecting the root and upper internal nodes of the learned Decision Tree reveals physically interpretable seismological rules:
1. **Root Split (`type_Mw <= 0.5`):** The root node partitions events by whether their reported magnitude scale is Moment Magnitude ($M_w$). In AFAD's operational protocol, $M_w$ is primarily computed for moderate-to-large waveforms with sufficient low-frequency signal-to-noise ratio. Consequently, when `type_Mw > 0.5` (right branch, $3.1\%$ of events), the class distribution shifts dramatically from `[0.67, 0.30, 0.03]` to `[0.01, 0.69, 0.30]`, immediately isolating `Minor (3.0-3.9)` and `Light+ (>=4.0)` events.
2. **Second-Level Splits (`depth <= 7.88 km` and `rms <= 0.48 s`):**
   - Along the left branch (`type_Mw <= 0.5`), very shallow events (`depth <= 7.88 km`, representing $71.0\%$ of the catalog) are predominantly `Micro (<3.0)` with $75\%$ purity (`value = [0.75, 0.24, 0.01]`), whereas deeper crustal events (`depth > 7.88 km`) exhibit a much higher proportion of `Minor` and `Light+` shocks (`value = [0.53, 0.42, 0.05]`).
   - Along the right branch (`type_Mw > 0.5`), the travel-time residual (`rms <= 0.48` vs. `rms > 0.48`) and epicentral `latitude <= 35.07°N` (Mediterranean subduction arc vs. continental Anatolia) separate `Minor (3.0-3.9)` events from `Light+ (>=4.0)` events (`value = [0.00, 0.47, 0.53]`).

In [6]:
# Visualize Top 10 Feature Importances (Decision Tree vs. Random Forest) & Spatial Test Predictions
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))

dt_imp = pd.Series(trained_models["Decision Tree"].feature_importances_, index=feature_names)
rf_imp = pd.Series(trained_models["Random Forest"].feature_importances_, index=feature_names)
top_feats = rf_imp.sort_values(ascending=True).tail(10).index

y_pos = np.arange(len(top_feats))
height = 0.38
axes[0].barh(y_pos - height/2, dt_imp[top_feats], height=height, label="Decision Tree", color="#3498db", alpha=0.9)
axes[0].barh(y_pos + height/2, rf_imp[top_feats], height=height, label="Random Forest", color="#e67e22", alpha=0.9)
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(top_feats, fontsize=9.5)
axes[0].set_xlabel("Relative Gini Feature Importance", fontsize=10)
axes[0].set_title("Top 10 Predictive Features (Decision Tree vs. Random Forest)", fontsize=12, fontweight="bold")
axes[0].legend(loc="lower right")
axes[0].grid(True, axis="x", linestyle="--", alpha=0.5)

gb_preds = test_predictions["Gradient Boosting"]
palette_map = {0: "#2b83ba", 1: "#fdae61", 2: "#d7191c"}
for c_idx, c_name in enumerate(class_names):
    mask = gb_preds == c_idx
    pt_size = 3 if c_idx == 0 else (6 if c_idx == 1 else 18)
    alpha_val = 0.25 if c_idx == 0 else (0.45 if c_idx == 1 else 0.85)
    axes[1].scatter(
        X_test_fe.loc[mask, "longitude"],
        X_test_fe.loc[mask, "latitude"],
        s=pt_size,
        alpha=alpha_val,
        c=palette_map[c_idx],
        label=f"Predicted: {c_name} (n={mask.sum():,})",
    )
axes[1].set_title("Spatial Distribution of Learned Predictions on Testing Set (20%)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Longitude (°E)", fontsize=10)
axes[1].set_ylabel("Latitude (°N)", fontsize=10)
axes[1].legend(loc="upper right", markerscale=2.0, fontsize=9)
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("../figures/fig15_feature_importance_and_spatial_map.png", bbox_inches="tight")
plt.show()

### Analysis of Feature Importances and Spatial Predictions (`fig15_feature_importance_and_spatial_map.png`)
1. **Gini Importance Comparison (Left Panel):**
   - The single **Decision Tree** concentrates heavily on raw `depth` ($17.5\%$), `type_Mw` ($13.3\%$), and `year` ($12.2\%$).
   - **Random Forest** distributes importance more evenly across correlated physical and spatial descriptors (`depth`, `year`, `log_depth`, `type_Mw`, `latitude`, `lat_lon_interaction`, `longitude`, and `province_seismic_index`) due to random feature subsampling (`max_features='sqrt'`) at each split, which decorrelates the individual trees and improves generalization.
2. **Tectonic Alignment of Test Predictions (Right Panel):**
   - Plotting the $30,688$ test set predictions geographically confirms that the learned classifier captures major Anatolian fault zones: `Light+ (>=4.0)` predictions (red markers) cluster distinctly along the **East Anatolian Fault Zone** (Kahramanmaraş–Malatya–Elazığ corridor), the **Aegean Extensional Province / Hellenic Arc** (southwest offshore), and the **North Anatolian Fault Zone**.

In [7]:
# Plot 2x2 Confusion Matrices on the 20% Testing Set for all 4 Classification Algorithms
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()

for idx, name in enumerate(models.keys()):
    cm = confusion_matrix(y_test, test_predictions[name])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100.0
    annot_matrix = np.empty(cm.shape, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot_matrix[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]:.1f}%)"

    sns.heatmap(
        cm_norm,
        annot=annot_matrix,
        fmt="",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        cbar=True,
        ax=axes[idx],
        vmin=0,
        vmax=100,
        annot_kws={"size": 9.5, "weight": "bold"},
    )
    acc = accuracy_score(y_test, test_predictions[name])
    wf1 = f1_score(y_test, test_predictions[name], average="weighted")
    axes[idx].set_title(f"{name}\n(Test Accuracy: {acc:.4f} | Weighted F1: {wf1:.4f})", fontsize=11.5, fontweight="bold")
    axes[idx].set_xlabel("Predicted Severity Class", fontsize=10)
    axes[idx].set_ylabel("Actual Severity Class", fontsize=10)

plt.suptitle("Testing Set (20%) Confusion Matrices Across Classification Algorithms", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../figures/fig16_confusion_matrices_comparison.png", bbox_inches="tight")
plt.show()

### Analysis of Confusion Matrices (`fig16_confusion_matrices_comparison.png`)
A detailed examination of the row-normalized confusion matrices on the held-out **20% Testing Set ($30,688$ samples)** reveals distinct behavioral differences across the four classifiers:
1. **`Micro (<3.0)` Class ($n = 20,532$):** All models achieve high recall on background micro-earthquakes, ranging from **$87.1\%$** (Decision Tree) to **$95.4\%$** (Logistic Regression). Notably, extreme misclassifications from `Micro (<3.0)` to `Light+ (>=4.0)` are virtually zero ($\le 0.1\%$ across all models; only $1$ sample out of $20,532$ in Random Forest and Gradient Boosting).
2. **`Minor (3.0-3.9)` Class ($n = 9,196$):** Because `Minor` earthquakes share continuous spatial and depth boundaries with upper-end `Micro` events ($M \approx 2.8$–$3.1$), linear **Logistic Regression** fails to separate them, misclassifying **$79.8\%$** ($7,341$ events) as `Micro` and achieving only **$17.4\%$** recall. In contrast, **Decision Tree** ($42.1\%$ recall, $3,872$ true positives) and **Gradient Boosting** ($39.0\%$ recall, $3,586$ true positives) capture more than double the true `Minor` events through non-linear partitioning.
3. **`Light+ (>=4.0)` Class ($n = 960$):** Both **Decision Tree** ($295$ true positives, **$30.7\%$ recall**) and **Gradient Boosting** ($291$ true positives, **$30.3\%$ recall**, with **$63.4\%$ precision**) identify nearly one-third of significant shocks without any magnitude or amplitude column in the feature set (they do rely on the magnitude-scale type `type_*` and the residual `rms`, which are indirect proxies; see the ablation in Section 5.5), whereas most remaining `Light+` events fall into the adjacent `Minor (3.0-3.9)` tier ($57.7\%$) rather than `Micro (<3.0)` ($12.0\%$).

In [8]:
# Comprehensive Performance Comparison Table (Accuracy, Precision, Recall, F1-Score) & Bar Chart
def evaluate_partition(preds_dict, y_true):
    rows = []
    for name, preds in preds_dict.items():
        rows.append({
            "Algorithm": name,
            "Accuracy": accuracy_score(y_true, preds),
            "Precision (Weighted)": precision_score(y_true, preds, average="weighted", zero_division=0),
            "Recall (Weighted)": recall_score(y_true, preds, average="weighted", zero_division=0),
            "F1-Score (Weighted)": f1_score(y_true, preds, average="weighted", zero_division=0),
            "Precision (Macro)": precision_score(y_true, preds, average="macro", zero_division=0),
            "Recall (Macro)": recall_score(y_true, preds, average="macro", zero_division=0),
            "F1-Score (Macro)": f1_score(y_true, preds, average="macro", zero_division=0),
        })
    return pd.DataFrame(rows)

val_df = evaluate_partition(val_predictions, y_val)
test_df = evaluate_partition(test_predictions, y_test)

print(val_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print()
print(test_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Plot Grouped Bar Charts Comparing Accuracy, Precision, Recall, and F1-Score on Testing Set
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
metric_cols = ["Accuracy", "Precision (Weighted)", "Recall (Weighted)", "F1-Score (Weighted)"]
short_labels = ["Accuracy", "Precision", "Recall", "F1-Score"]
model_names = list(models.keys())
x = np.arange(len(short_labels))
width = 0.20
colors = ["#2980b9", "#27ae60", "#8e44ad", "#c0392b"]

for i, m_name in enumerate(model_names):
    row = test_df[test_df["Algorithm"] == m_name].iloc[0]
    vals = [row[mc] for mc in metric_cols]
    bars = axes[0].bar(x + (i - 1.5) * width, vals, width, label=m_name, color=colors[i], alpha=0.92)
    for b in bars:
        axes[0].annotate(
            f"{b.get_height():.3f}",
            (b.get_x() + b.get_width() / 2.0, b.get_height()),
            ha="center", va="bottom", fontsize=7.5, xytext=(0, 2), textcoords="offset points"
        )

axes[0].set_xticks(x)
axes[0].set_xticklabels(short_labels, fontsize=10.5, fontweight="bold")
axes[0].set_ylim(0.50, 0.90)
axes[0].set_ylabel("Score on Testing Set (20%)", fontsize=10.5)
axes[0].set_title("Overall Classification Performance Comparison (Weighted Metrics)", fontsize=12, fontweight="bold")
axes[0].legend(loc="lower right", fontsize=9)
axes[0].grid(True, axis="y", linestyle="--", alpha=0.6)

macro_cols = ["Accuracy", "Precision (Macro)", "Recall (Macro)", "F1-Score (Macro)"]
for i, m_name in enumerate(model_names):
    row = test_df[test_df["Algorithm"] == m_name].iloc[0]
    vals = [row[mc] for mc in macro_cols]
    bars = axes[1].bar(x + (i - 1.5) * width, vals, width, label=m_name, color=colors[i], alpha=0.92)
    for b in bars:
        axes[1].annotate(
            f"{b.get_height():.3f}",
            (b.get_x() + b.get_width() / 2.0, b.get_height()),
            ha="center", va="bottom", fontsize=7.5, xytext=(0, 2), textcoords="offset points"
        )

axes[1].set_xticks(x)
axes[1].set_xticklabels(["Accuracy", "Macro Precision", "Macro Recall", "Macro F1-Score"], fontsize=10.5, fontweight="bold")
axes[1].set_ylim(0.25, 0.90)
axes[1].set_ylabel("Score on Testing Set (20%)", fontsize=10.5)
axes[1].set_title("Class-Balanced Performance Comparison (Macro-Averaged Metrics)", fontsize=12, fontweight="bold")
axes[1].legend(loc="upper right", fontsize=9)
axes[1].grid(True, axis="y", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig("../figures/fig17_classification_metrics_comparison.png", bbox_inches="tight")
plt.show()

          Algorithm  Accuracy  Precision (Weighted)  Recall (Weighted)  F1-Score (Weighted)  Precision (Macro)  Recall (Macro)  F1-Score (Macro)
      Decision Tree    0.7211                0.7027             0.7211               0.7071             0.6146          0.5287            0.5573
      Random Forest    0.7474                0.7289             0.7474               0.7176             0.6830          0.5138            0.5519
Logistic Regression    0.6957                0.6524             0.6957               0.6364             0.5866          0.4620            0.4760
  Gradient Boosting    0.7527                0.7351             0.7527               0.7269             0.6814          0.5325            0.5711

          Algorithm  Accuracy  Precision (Weighted)  Recall (Weighted)  F1-Score (Weighted)  Precision (Macro)  Recall (Macro)  F1-Score (Macro)
      Decision Tree    0.7184                0.6992             0.7184               0.7037             0.6085          0.5330   

## 5. Additional Experiments: Tuning, Class Imbalance, Ablation and Uncertainty

The experiments below extend the core pipeline above **without changing any of its results**. Hyperparameters and model variants are selected on the 20% *validation* set; the 20% *testing* set is only used for final reporting (Sections 5.1-5.3, 5.7, 5.8). The diagnostic experiments in Sections 5.5 and 5.6 use the validation set so that the testing set stays untouched.

| Section | Question answered |
|---|---|
| 5.1 | Per-class precision / recall / F1 and a naive majority-class baseline |
| 5.2 | Do Random Forest, Gradient Boosting and Logistic Regression improve when tuned on the validation set? |
| 5.3 | Can cost-sensitive (balanced) training recover more `Light+` events, and at what price in precision? |
| 5.4 | A compact depth-3 decision tree that can be displayed completely |
| 5.5 | How strongly does the classifier depend on catalogue metadata (`type_*`, `rms`)? |
| 5.6 | Permutation importance of the best model (less biased than Gini importance) |
| 5.7 | Bootstrap 95% confidence intervals on the testing set |
| 5.8 | Sensitivity of the results to the in-sample target encoding of `province_seismic_index` |

In [9]:
# 5.1 Per-class precision / recall / F1 (testing set), naive baseline and CSV export
from sklearn.dummy import DummyClassifier

os.makedirs("../results", exist_ok=True)

dummy = DummyClassifier(strategy="most_frequent").fit(X_train_fe, y_train)
dummy_test_pred = dummy.predict(X_test_fe)

baseline_row = evaluate_partition({"Majority-class baseline": dummy_test_pred}, y_test)
test_df_with_baseline = pd.concat([test_df, baseline_row], ignore_index=True)
print("=== Testing-set metrics including the naive majority-class baseline ===")
print(test_df_with_baseline.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

per_class_rows = []
for name, preds in {**test_predictions, "Majority-class baseline": dummy_test_pred}.items():
    p = precision_score(y_test, preds, labels=[0, 1, 2], average=None, zero_division=0)
    r = recall_score(y_test, preds, labels=[0, 1, 2], average=None, zero_division=0)
    f = f1_score(y_test, preds, labels=[0, 1, 2], average=None, zero_division=0)
    for k, cname in enumerate(class_names):
        per_class_rows.append({
            "Algorithm": name, "Class": cname, "Precision": p[k], "Recall": r[k],
            "F1-Score": f[k], "Support": int((y_test == k).sum()),
        })
per_class_df = pd.DataFrame(per_class_rows)
print("\n=== Per-class testing-set metrics ===")
print(per_class_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

val_df.to_csv("../results/cls_validation_metrics.csv", index=False)
test_df_with_baseline.to_csv("../results/cls_test_metrics.csv", index=False)
per_class_df.to_csv("../results/cls_per_class_test_metrics.csv", index=False)
print("\nSaved CSV files to ../results/")

=== Testing-set metrics including the naive majority-class baseline ===
              Algorithm  Accuracy  Precision (Weighted)  Recall (Weighted)  F1-Score (Weighted)  Precision (Macro)  Recall (Macro)  F1-Score (Macro)
          Decision Tree    0.7184                0.6992             0.7184               0.7037             0.6085          0.5330            0.5593
          Random Forest    0.7510                0.7345             0.7510               0.7214             0.6922          0.5221            0.5616
    Logistic Regression    0.6989                0.6593             0.6989               0.6388             0.5869          0.4656            0.4778
      Gradient Boosting    0.7539                0.7376             0.7539               0.7282             0.6909          0.5437            0.5835
Majority-class baseline    0.6691                0.4476             0.6691               0.5364             0.2230          0.3333            0.2672

=== Per-class testing-set metrics

In [10]:
# 5.2 Validation-set tuning of the remaining three algorithms (the testing set is NOT used for selection)
import itertools
import time

grids = {
    "Random Forest": (
        [dict(max_depth=d, min_samples_leaf=l) for d, l in itertools.product([12, 16, 24], [5, 15])],
        lambda p: RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=42, **p),
        False,
    ),
    "Gradient Boosting": (
        [dict(learning_rate=lr, max_depth=d) for lr, d in itertools.product([0.04, 0.08, 0.15], [6, 10])],
        lambda p: HistGradientBoostingClassifier(max_iter=200, min_samples_leaf=25, random_state=42, **p),
        False,
    ),
    "Logistic Regression": (
        [dict(C=c) for c in [0.01, 0.1, 1.0, 10.0]],
        lambda p: LogisticRegression(max_iter=500, random_state=42, **p),
        True,
    ),
}

tuning_rows, best_params, tuned_models = [], {}, {}
for name, (grid, factory, use_scaled) in grids.items():
    Xtr = X_train_scaled if use_scaled else X_train_fe
    Xva = X_val_scaled if use_scaled else X_val_fe
    best_score, best_p = -1.0, None
    for p in grid:
        t0 = time.perf_counter()
        clf = factory(p).fit(Xtr, y_train)
        pred = clf.predict(Xva)
        wf1 = f1_score(y_val, pred, average="weighted")
        tuning_rows.append({
            "Algorithm": name, "Params": str(p), "Val Accuracy": accuracy_score(y_val, pred),
            "Val Weighted F1": wf1, "Val Macro F1": f1_score(y_val, pred, average="macro"),
            "Seconds": time.perf_counter() - t0,
        })
        if wf1 > best_score:
            best_score, best_p = wf1, p
    best_params[name] = best_p
    tuned_models[name] = factory(best_p).fit(Xtr, y_train)

tuning_df = pd.DataFrame(tuning_rows)
print("=== Validation-set grid search (selection criterion: validation weighted F1) ===")
print(tuning_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nSelected hyperparameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

tuned_test_preds = {"Decision Tree": test_predictions["Decision Tree"]}   # tuned in the cell above
for name, clf in tuned_models.items():
    use_scaled = grids[name][2]
    tuned_test_preds[name] = clf.predict(X_test_scaled if use_scaled else X_test_fe)

cols = ["Algorithm", "Accuracy", "F1-Score (Weighted)", "F1-Score (Macro)"]
tuned_test_df = evaluate_partition(tuned_test_preds, y_test)
cmp_tuned = test_df[cols].merge(tuned_test_df[cols], on="Algorithm", suffixes=(" | default", " | tuned"))
print("\n=== Testing set: default vs. validation-tuned hyperparameters ===")
print(cmp_tuned.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

tuning_df.to_csv("../results/cls_validation_grid_search.csv", index=False)
cmp_tuned.to_csv("../results/cls_default_vs_tuned_test.csv", index=False)

=== Validation-set grid search (selection criterion: validation weighted F1) ===
          Algorithm                                    Params  Val Accuracy  Val Weighted F1  Val Macro F1  Seconds
      Random Forest  {'max_depth': 12, 'min_samples_leaf': 5}        0.7432           0.7087        0.5383   4.3185
      Random Forest {'max_depth': 12, 'min_samples_leaf': 15}        0.7428           0.7077        0.5371   4.2861
      Random Forest  {'max_depth': 16, 'min_samples_leaf': 5}        0.7493           0.7200        0.5591   5.3338
      Random Forest {'max_depth': 16, 'min_samples_leaf': 15}        0.7472           0.7168        0.5482   5.0804
      Random Forest  {'max_depth': 24, 'min_samples_leaf': 5}        0.7506           0.7256        0.5695   6.5610
      Random Forest {'max_depth': 24, 'min_samples_leaf': 15}        0.7481           0.7197        0.5520   5.8080
  Gradient Boosting   {'learning_rate': 0.04, 'max_depth': 6}        0.7506           0.7241        0.5665 


=== Testing set: default vs. validation-tuned hyperparameters ===
          Algorithm  Accuracy | default  F1-Score (Weighted) | default  F1-Score (Macro) | default  Accuracy | tuned  F1-Score (Weighted) | tuned  F1-Score (Macro) | tuned
      Decision Tree              0.7184                         0.7037                      0.5593            0.7184                       0.7037                    0.5593
      Random Forest              0.7510                         0.7214                      0.5616            0.7526                       0.7277                    0.5766
Logistic Regression              0.6989                         0.6388                      0.4778            0.6990                       0.6388                    0.4743
  Gradient Boosting              0.7539                         0.7282                      0.5835            0.7539                       0.7282                    0.5835


In [11]:
# 5.3 Class-imbalance experiment: cost-sensitive ("balanced") training of all four algorithms
from sklearn.utils.class_weight import compute_sample_weight

sw_train = compute_sample_weight("balanced", y_train)
bal_models = {
    "Decision Tree": (DecisionTreeClassifier(max_depth=best_dt_depth, min_samples_leaf=25, class_weight="balanced", random_state=42), False, None),
    "Random Forest": (RandomForestClassifier(n_estimators=150, max_depth=16, min_samples_leaf=10, class_weight="balanced_subsample", n_jobs=-1, random_state=42), False, None),
    "Logistic Regression": (LogisticRegression(max_iter=500, C=1.0, class_weight="balanced", random_state=42), True, None),
    "Gradient Boosting": (HistGradientBoostingClassifier(max_iter=200, max_depth=10, learning_rate=0.08, min_samples_leaf=25, random_state=42), False, sw_train),
}

bal_test_preds = {}
for name, (clf, use_scaled, sw) in bal_models.items():
    Xtr = X_train_scaled if use_scaled else X_train_fe
    Xte = X_test_scaled if use_scaled else X_test_fe
    clf.fit(Xtr, y_train, sample_weight=sw)
    bal_test_preds[name] = clf.predict(Xte)

cmp_rows = []
for name in bal_models:
    for variant, preds in (("Default", test_predictions[name]), ("Balanced", bal_test_preds[name])):
        p = precision_score(y_test, preds, labels=[0, 1, 2], average=None, zero_division=0)
        r = recall_score(y_test, preds, labels=[0, 1, 2], average=None, zero_division=0)
        f = f1_score(y_test, preds, labels=[0, 1, 2], average=None, zero_division=0)
        cmp_rows.append({
            "Algorithm": name, "Variant": variant, "Accuracy": accuracy_score(y_test, preds),
            "Macro F1": f.mean(), "Minor Recall": r[1],
            "Light+ Precision": p[2], "Light+ Recall": r[2], "Light+ F1": f[2],
        })
bal_df = pd.DataFrame(cmp_rows)
print("=== Testing set: default vs. class-balanced training ===")
print(bal_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
bal_df.to_csv("../results/cls_class_balance_experiment.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
for ax, metric in zip(axes, ["Light+ Recall", "Light+ Precision", "Macro F1"]):
    sns.barplot(data=bal_df, x="Algorithm", y=metric, hue="Variant", ax=ax, palette=["#7f8c8d", "#c0392b"])
    for cont in ax.containers:
        ax.bar_label(cont, fmt="%.3f", fontsize=8, padding=2)
    ax.set_title(f"{metric}: default vs. class-balanced training", fontsize=11.5, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylim(0, bal_df[metric].max() * 1.30)
    ax.legend(title="Variant", loc="upper center", ncol=2, fontsize=9)
    ax.tick_params(axis="x", labelrotation=15)
    ax.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("../figures/fig24_class_balance_experiment.png", bbox_inches="tight")
plt.show()

=== Testing set: default vs. class-balanced training ===
          Algorithm  Variant  Accuracy  Macro F1  Minor Recall  Light+ Precision  Light+ Recall  Light+ F1
      Decision Tree  Default    0.7184    0.5593        0.4211            0.5008         0.3073     0.3809
      Decision Tree Balanced    0.6104    0.5036        0.4808            0.2109         0.7844     0.3324
      Random Forest  Default    0.7510    0.5616        0.3683            0.6419         0.2521     0.3620
      Random Forest Balanced    0.7013    0.6028        0.5279            0.3696         0.7156     0.4874
Logistic Regression  Default    0.6989    0.4778        0.1740            0.5000         0.2687     0.3496
Logistic Regression Balanced    0.6139    0.5029        0.3810            0.2513         0.7812     0.3802
  Gradient Boosting  Default    0.7539    0.5835        0.3900            0.6340         0.3031     0.4101
  Gradient Boosting Balanced    0.6684    0.5674        0.5283            0.2927       

In [12]:
# 5.4 Compact decision tree (max_depth = 3) that can be displayed completely
small_dt = DecisionTreeClassifier(max_depth=3, min_samples_leaf=25, criterion="gini", random_state=42)
small_dt.fit(X_train_fe, y_train)
sm_val, sm_test = small_dt.predict(X_val_fe), small_dt.predict(X_test_fe)
print("Compact tree (max_depth=3)")
print(f"  Validation: accuracy={accuracy_score(y_val, sm_val):.4f} | weighted F1={f1_score(y_val, sm_val, average='weighted'):.4f} | macro F1={f1_score(y_val, sm_val, average='macro'):.4f}")
print(f"  Testing   : accuracy={accuracy_score(y_test, sm_test):.4f} | weighted F1={f1_score(y_test, sm_test, average='weighted'):.4f} | macro F1={f1_score(y_test, sm_test, average='macro'):.4f}")
print()
print(export_text(small_dt, feature_names=feature_names))

fig, ax = plt.subplots(figsize=(22, 8))
plot_tree(
    small_dt, feature_names=feature_names, class_names=class_names,
    filled=True, rounded=True, proportion=True, fontsize=9, precision=2, ax=ax,
)
ax.set_title("Compact Decision Tree Classifier (max_depth = 3, fully displayed)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../figures/fig23_compact_decision_tree.png", bbox_inches="tight")
plt.show()

Compact tree (max_depth=3)
  Validation: accuracy=0.6973 | weighted F1=0.5983 | macro F1=0.4507
  Testing   : accuracy=0.7001 | weighted F1=0.6027 | macro F1=0.4631

|--- type_Mw <= 0.50
|   |--- depth <= 7.88
|   |   |--- type_Other <= 0.50
|   |   |   |--- class: 0
|   |   |--- type_Other >  0.50
|   |   |   |--- class: 2
|   |--- depth >  7.88
|   |   |--- type_Other <= 0.50
|   |   |   |--- class: 0
|   |   |--- type_Other >  0.50
|   |   |   |--- class: 2
|--- type_Mw >  0.50
|   |--- log_rms <= 0.40
|   |   |--- depth <= 7.57
|   |   |   |--- class: 1
|   |   |--- depth >  7.57
|   |   |   |--- class: 1
|   |--- log_rms >  0.40
|   |   |--- latitude <= 35.07
|   |   |   |--- class: 2
|   |   |--- latitude >  35.07
|   |   |   |--- class: 1



In [13]:
# 5.5 Feature-group ablation (evaluated on the VALIDATION set; Gradient Boosting with the default configuration)
def make_gb():
    return HistGradientBoostingClassifier(max_iter=200, max_depth=10, learning_rate=0.08, min_samples_leaf=25, random_state=42)

type_cols_cls = [c for c in feature_names if c.startswith("type_")]
ablation_sets = {
    "All 19 features (reference)": [],
    "Without magnitude-scale type (type_*)": type_cols_cls,
    "Without type_* and rms / log_rms": type_cols_cls + ["rms", "log_rms"],
    "Without temporal features (year, month, hour)": [c for c in feature_names if c.startswith(("month_", "hour_")) or c == "year"],
    "Location and depth only": [c for c in feature_names if c not in
                                ["latitude", "longitude", "depth", "log_depth", "lat_lon_interaction", "province_freq", "province_seismic_index"]],
}

abl_rows = []
for label, drop in ablation_sets.items():
    keep = [c for c in feature_names if c not in drop]
    clf = make_gb().fit(X_train_fe[keep], y_train)
    pv = clf.predict(X_val_fe[keep])
    r = recall_score(y_val, pv, labels=[0, 1, 2], average=None, zero_division=0)
    f = f1_score(y_val, pv, labels=[0, 1, 2], average=None, zero_division=0)
    abl_rows.append({
        "Scenario": label, "n features": len(keep), "Val Accuracy": accuracy_score(y_val, pv),
        "Val Weighted F1": f1_score(y_val, pv, average="weighted"), "Val Macro F1": f.mean(),
        "Val Light+ Recall": r[2], "Val Light+ F1": f[2],
    })
abl_df = pd.DataFrame(abl_rows)
print("=== Feature-group ablation (validation set) ===")
print(abl_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
abl_df.to_csv("../results/cls_feature_ablation_validation.csv", index=False)

plot_df = abl_df.melt(id_vars="Scenario", value_vars=["Val Weighted F1", "Val Macro F1", "Val Light+ Recall"],
                      var_name="Metric", value_name="Score")
fig, ax = plt.subplots(figsize=(12, 5.5))
sns.barplot(data=plot_df, y="Scenario", x="Score", hue="Metric", ax=ax, palette=["#2980b9", "#27ae60", "#c0392b"])
for cont in ax.containers:
    ax.bar_label(cont, fmt="%.3f", fontsize=8, padding=2)
ax.set_title("Feature-group ablation (Gradient Boosting, validation set)", fontsize=12, fontweight="bold")
ax.set_ylabel("")
ax.legend(title="Metric", loc="upper left", bbox_to_anchor=(1.01, 1.0))
ax.set_xlim(0, 0.85)
ax.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("../figures/fig26_feature_ablation.png", bbox_inches="tight")
plt.show()

=== Feature-group ablation (validation set) ===
                                     Scenario  n features  Val Accuracy  Val Weighted F1  Val Macro F1  Val Light+ Recall  Val Light+ F1
                  All 19 features (reference)          19        0.7527           0.7269        0.5711             0.2687         0.3736
        Without magnitude-scale type (type_*)          14        0.7180           0.6793        0.4349             0.0302         0.0566
             Without type_* and rms / log_rms          12        0.7158           0.6765        0.4318             0.0292         0.0545
Without temporal features (year, month, hour)          14        0.7388           0.7087        0.5423             0.2323         0.3336
                      Location and depth only           7        0.6919           0.6449        0.3926             0.0115         0.0222


In [14]:
# 5.6 Permutation importance of the Gradient Boosting model (validation set, drop in macro F1)
from sklearn.inspection import permutation_importance

gb_model = trained_models["Gradient Boosting"]
perm = permutation_importance(gb_model, X_val_fe, y_val, scoring="f1_macro", n_repeats=5, random_state=42, n_jobs=-1)
perm_df = pd.DataFrame({
    "Feature": feature_names, "Mean drop in macro F1": perm.importances_mean, "Std": perm.importances_std,
}).sort_values("Mean drop in macro F1", ascending=False)
print(perm_df.head(12).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
perm_df.to_csv("../results/cls_permutation_importance_gb_validation.csv", index=False)

top = perm_df.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top["Feature"], top["Mean drop in macro F1"], xerr=top["Std"], color="#c0392b", alpha=0.9, capsize=3)
ax.set_xlabel("Mean decrease in validation macro F1 when the feature is shuffled")
ax.set_title("Permutation Importance (Gradient Boosting, validation set)", fontsize=12, fontweight="bold")
ax.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("../figures/fig25_permutation_importance.png", bbox_inches="tight")
plt.show()

               Feature  Mean drop in macro F1    Std
                  year                 0.0898 0.0029
            type_Other                 0.0800 0.0009
               type_Md                 0.0516 0.0050
                 depth                 0.0468 0.0022
               type_Mw                 0.0362 0.0011
              latitude                 0.0147 0.0036
             longitude                 0.0137 0.0020
               type_ML                 0.0127 0.0027
province_seismic_index                 0.0122 0.0023
   lat_lon_interaction                 0.0100 0.0016
                   rms                 0.0094 0.0024
               type_Ml                 0.0065 0.0010


In [15]:
# 5.7 Bootstrap 95% confidence intervals on the testing set (1,000 paired resamples)
B = 1000
rng_b = np.random.default_rng(42)
yt = np.asarray(y_test, dtype=np.int64)
preds_arr = {k: np.asarray(v, dtype=np.int64) for k, v in test_predictions.items()}
metric_labels = ["Accuracy", "Macro F1", "Light+ Recall", "Light+ F1"]

def cm_scores(yt_b, yp_b):
    cm = np.bincount(3 * yt_b + yp_b, minlength=9).reshape(3, 3).astype(float)
    tp = np.diag(cm)
    prec = tp / np.maximum(cm.sum(axis=0), 1.0)
    rec = tp / np.maximum(cm.sum(axis=1), 1.0)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    return np.array([tp.sum() / cm.sum(), f1.mean(), rec[2], f1[2]])

boot = {k: np.zeros((B, 4)) for k in preds_arr}
for b in range(B):
    idx = rng_b.integers(0, len(yt), len(yt))
    for k in preds_arr:
        boot[k][b] = cm_scores(yt[idx], preds_arr[k][idx])

rows = []
for k in preds_arr:
    point = cm_scores(yt, preds_arr[k])
    lo, hi = np.percentile(boot[k], [2.5, 97.5], axis=0)
    for j, lab in enumerate(metric_labels):
        rows.append({"Algorithm": k, "Metric": lab, "Point estimate": point[j], "CI 2.5%": lo[j], "CI 97.5%": hi[j]})
ci_df = pd.DataFrame(rows)
print("=== Testing-set bootstrap 95% confidence intervals ===")
print(ci_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
ci_df.to_csv("../results/cls_bootstrap_ci_test.csv", index=False)

print("\n=== Paired bootstrap: Gradient Boosting minus Random Forest ===")
for j, lab in enumerate(metric_labels):
    d = boot["Gradient Boosting"][:, j] - boot["Random Forest"][:, j]
    lo, hi = np.percentile(d, [2.5, 97.5])
    verdict = "interval excludes 0" if (lo > 0 or hi < 0) else "interval contains 0 (difference not clearly distinguishable from noise)"
    print(f"  {lab:<14}: mean diff = {d.mean():+.4f} | 95% CI [{lo:+.4f}, {hi:+.4f}] -> {verdict}")

=== Testing-set bootstrap 95% confidence intervals ===
          Algorithm        Metric  Point estimate  CI 2.5%  CI 97.5%
      Decision Tree      Accuracy          0.7184   0.7135    0.7231
      Decision Tree      Macro F1          0.5593   0.5485    0.5701
      Decision Tree Light+ Recall          0.3073   0.2798    0.3365
      Decision Tree     Light+ F1          0.3809   0.3510    0.4106
      Random Forest      Accuracy          0.7510   0.7461    0.7559
      Random Forest      Macro F1          0.5616   0.5490    0.5738
      Random Forest Light+ Recall          0.2521   0.2251    0.2817
      Random Forest     Light+ F1          0.3620   0.3268    0.3959
Logistic Regression      Accuracy          0.6989   0.6938    0.7039
Logistic Regression      Macro F1          0.4778   0.4669    0.4891
Logistic Regression Light+ Recall          0.2687   0.2424    0.2978
Logistic Regression     Light+ F1          0.3496   0.3197    0.3802
  Gradient Boosting      Accuracy          0.753

In [16]:
# 5.8 Sensitivity check: out-of-fold (OOF) target encoding of `province_seismic_index`
# In the main pipeline the province mean-target encoding is estimated on the training split and then also applied to the
# training rows themselves (in-sample). Here each training row is encoded with a table estimated WITHOUT its own fold.
from sklearn.model_selection import KFold

def oof_province_index(prov_train, y_train_arr, prov_others, n_splits=5, smoothing=20):
    gm = float(np.mean(y_train_arr))

    def table(p, y):
        s = pd.DataFrame({"p": p.to_numpy(), "y": y}).groupby("p")["y"].agg(["sum", "count"])
        return (s["sum"] + smoothing * gm) / (s["count"] + smoothing)

    prov_train = prov_train.reset_index(drop=True)
    oof = np.zeros(len(prov_train))
    for a, b in KFold(n_splits, shuffle=True, random_state=42).split(prov_train):
        oof[b] = prov_train.iloc[b].map(table(prov_train.iloc[a], y_train_arr[a])).fillna(gm).to_numpy()
    full = table(prov_train, y_train_arr)
    return oof, [p.map(full).fillna(gm).to_numpy() for p in prov_others]

oof_train, (oof_val, oof_test) = oof_province_index(
    X_train_raw["province"], np.asarray(y_train), [X_val_raw["province"], X_test_raw["province"]]
)
X_train_oof, X_val_oof, X_test_oof = X_train_fe.copy(), X_val_fe.copy(), X_test_fe.copy()
X_train_oof["province_seismic_index"] = oof_train
X_val_oof["province_seismic_index"] = oof_val
X_test_oof["province_seismic_index"] = oof_test

def fit_eval(make_model, Xtr, Xva, Xte):
    m = make_model().fit(Xtr, y_train)
    pv, pt = m.predict(Xva), m.predict(Xte)
    return {
        "Val Weighted F1": f1_score(y_val, pv, average="weighted"),
        "Test Accuracy": accuracy_score(y_test, pt),
        "Test Weighted F1": f1_score(y_test, pt, average="weighted"),
        "Test Macro F1": f1_score(y_test, pt, average="macro"),
        "Test Light+ Recall": recall_score(y_test, pt, labels=[2], average=None, zero_division=0)[0],
    }

makers = {
    "Decision Tree": lambda: DecisionTreeClassifier(max_depth=best_dt_depth, min_samples_leaf=25, criterion="gini", random_state=42),
    "Gradient Boosting": make_gb,
}
oof_rows = []
for name, mk in makers.items():
    a = fit_eval(mk, X_train_fe, X_val_fe, X_test_fe)
    b = fit_eval(mk, X_train_oof, X_val_oof, X_test_oof)
    oof_rows.append({"Algorithm": name, "Encoding": "In-sample (main pipeline)", **a})
    oof_rows.append({"Algorithm": name, "Encoding": "5-fold out-of-fold", **b})
oof_df = pd.DataFrame(oof_rows)
print("=== Effect of out-of-fold province target encoding ===")
print(oof_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
oof_df.to_csv("../results/cls_oof_encoding_sensitivity.csv", index=False)
print("\nIf the differences are far smaller than the bootstrap intervals of Section 5.7, the in-sample encoding does not materially affect the conclusions.")

=== Effect of out-of-fold province target encoding ===
        Algorithm                  Encoding  Val Weighted F1  Test Accuracy  Test Weighted F1  Test Macro F1  Test Light+ Recall
    Decision Tree In-sample (main pipeline)           0.7071         0.7184            0.7037         0.5593              0.3073
    Decision Tree        5-fold out-of-fold           0.7054         0.7181            0.7021         0.5485              0.2792
Gradient Boosting In-sample (main pipeline)           0.7269         0.7539            0.7282         0.5835              0.3031
Gradient Boosting        5-fold out-of-fold           0.7251         0.7548            0.7301         0.5841              0.3021

If the differences are far smaller than the bootstrap intervals of Section 5.7, the in-sample encoding does not materially affect the conclusions.


---

## 6. Comprehensive Discussions on Classification Findings

### 6.1. Overall Model Ranking and Quantitative Comparison
Across both the **20% Validation Set ($30,688$ events)** and the **20% Testing Set ($30,688$ events)**, the four classification algorithms rank consistently on accuracy and on the weighted and macro F1-scores (the exceptions are macro recall, where the Decision Tree exceeds Random Forest, and macro precision, where Random Forest is marginally higher than Gradient Boosting):

| Algorithm | Test Accuracy | Test Precision (Weighted) | Test Recall (Weighted) | Test F1-Score (Weighted) | Test Precision (Macro) | Test Recall (Macro) | Test F1-Score (Macro) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **1. Gradient Boosting** | **0.7539** | **0.7376** | **0.7539** | **0.7282** | 0.6909 | **0.5437** | **0.5835** |
| **2. Random Forest** | 0.7510 | 0.7345 | 0.7510 | 0.7214 | **0.6922** | 0.5221 | 0.5616 |
| **3. Decision Tree** | 0.7184 | 0.6992 | 0.7184 | 0.7037 | 0.6085 | 0.5330 | 0.5593 |
| **4. Logistic Regression** | 0.6989 | 0.6593 | 0.6989 | 0.6388 | 0.5869 | 0.4656 | 0.4778 |

1. **Superiority of Non-Linear Tree Ensembles (Gradient Boosting & Random Forest):**
   - **Gradient Boosting (`HistGradientBoostingClassifier`)** achieves the highest overall performance with **$75.39\%$ Accuracy**, **$0.7376$ Weighted Precision**, **$0.7539$ Weighted Recall**, **$0.7282$ Weighted F1-score**, and **$0.5835$ Macro F1-score**. Because gradient boosting sequentially fits additive trees to the negative gradient of the multinomial log-loss, it iteratively focuses on hard-to-classify boundary events between `Micro (<3.0)`, `Minor (3.0-3.9)`, and `Light+ (>=4.0)`.
   - **Random Forest** closely follows with **$75.10\%$ Accuracy** and **$0.7214$ Weighted F1-score**, outperforming the single Decision Tree by **$+3.26\%$ in Accuracy** and **$+3.53\%$ in Weighted Precision**. Averaging $150$ decorrelated trees substantially reduces the variance of individual axis-aligned splits, cutting `Micro`$\rightarrow$`Minor` confusions from $2,641$ (Decision Tree) down to $1,112$ (Random Forest); counting every sample wrongly predicted as `Minor`, false positives fall from $3,181$ ($2,641 + 540$) to $1,709$ ($1,112 + 597$).

2. **Single Decision Tree vs. Ensemble Methods (Interpretability vs. Variance Trade-off):**
   - The tuned **Decision Tree (`max_depth=20`, `min_samples_leaf=25`)** attains **$71.84\%$ Test Accuracy** and **$0.7037$ Weighted F1-score** ($0.5593$ Macro F1). Notably, its **Macro Recall ($0.5330$)** actually slightly exceeds that of Random Forest ($0.5221$), because an unbagged deeper tree forms sharper localized leaf regions for the minority `Minor` ($42.1\%$ recall) and `Light+` ($30.7\%$ recall) classes, albeit at the expense of lower precision ($54.9\%$ on `Minor` and $50.1\%$ on `Light+`).
   - From a scientific and operational standpoint, the Decision Tree provides transparent, human-verifiable decision rules (`fig14_decision_tree_visualization.png`), revealing how measurement scale type (`type_Mw`, `type_ML`), hypocenter `depth`, waveform `rms`, and tectonic coordinates interact hierarchically.

3. **Why Linear Logistic Regression Underperforms on Seismic Catalog Data:**
   - **Logistic Regression** yields the lowest performance across all metrics (**$69.89\%$ Accuracy**, **$0.6388$ Weighted F1**, and **$0.4778$ Macro F1**). While its accuracy appears superficially close to $70\%$, inspecting its per-class recall (`fig16_confusion_matrices_comparison.png`) exposes a critical failure mode: it achieves $95.4\%$ recall on the majority `Micro (<3.0)` class by defaulting almost all ambiguous observations to the majority class, resulting in only **$17.4\%$ recall ($F_1 = 0.2633$)** on `Minor (3.0-3.9)` earthquakes.
   - Seismological fault geometries across Anatolia (e.g., curved arc segments along the North Anatolian, East Anatolian, and Hellenic subduction zones) are inherently non-linear in $(\text{latitude}, \text{longitude}, \text{depth})$ space; a linear hyperplane cannot separate high-magnitude fault corridors from surrounding micro-seismic swarms.

### 6.2. Critical Role of Precision, Recall, and F1-Score Under Gutenberg–Richter Class Imbalance
- Because seismic catalogs follow the exponential Gutenberg–Richter distribution ($66.90\%$ `Micro`, $29.97\%$ `Minor`, $3.13\%$ `Light+`), **Accuracy alone is an inadequate evaluation metric**: a trivial majority-class classifier predicting `Micro (<3.0)` for every event would achieve $66.90\%$ accuracy while yielding $0.00$ recall on significant earthquakes and a Macro F1-score of only $0.2672$.
- Using **Weighted F1** ($0.7282$ for Gradient Boosting) and **Macro F1** ($0.5835$ for Gradient Boosting, more than double the naive baseline of $0.2672$) quantifies both catalog-wide reliability and minority-class sensitivity.
- Furthermore, validation and test metrics agree closely: accuracy and weighted F1 differ by at most $0.0038$ for every algorithm (e.g., Gradient Boosting Validation Weighted F1 $= 0.7269$ vs. Test Weighted F1 $= 0.7282$) and all macro-averaged metrics by at most $0.0124$ (Gradient Boosting Macro F1: $0.5711$ vs. $0.5835$); the larger macro-level gap reflects the small `Light+` support ($960$ samples per partition, see the bootstrap intervals in Section 5.7). This indicates that the stratified **60% / 20% / 20% partitioning** generalizes to unseen seismic events. Remaining caveats: the in-sample province target encoding (Section 5.8) and the dependence on magnitude-scale metadata (Section 5.5).

### 6.3. Findings of the Additional Experiments (Section 5)
1. **Tuning all algorithms on the validation set (Section 5.2).** The default configuration of Gradient Boosting was already the best of its grid (`learning_rate=0.08`, `max_depth=10`), so its results are unchanged. For Logistic Regression the regularisation strength hardly matters (validation weighted F1 $0.634$–$0.637$ for $C \in \{0.01, \dots, 10\}$), which confirms that its weakness is the linear decision boundary, not over- or under-regularisation. Random Forest benefits from deeper trees: the selected configuration (`max_depth=24`, `min_samples_leaf=5`) raises the test weighted F1 from $0.7214$ to $0.7277$ and the macro F1 from $0.5616$ to $0.5766$. Because this value lies at the edge of the grid, a slightly larger grid could improve Random Forest further. The ranking *Gradient Boosting > Random Forest > Decision Tree > Logistic Regression* on weighted F1 is preserved after tuning, but the gap between the two ensembles shrinks to $0.0005$.
2. **Class imbalance (Section 5.3, Fig. 24).** Cost-sensitive ("balanced") training changes the operating point dramatically: `Light+` recall rises from $0.25$–$0.31$ to $0.72$–$0.79$ for every algorithm, while `Light+` precision falls from $0.50$–$0.64$ to $0.21$–$0.37$ and accuracy drops by $5$–$11$ percentage points. The best compromise is the balanced Random Forest, which reaches the highest macro F1 of all experiments ($0.6028$) and a `Light+` F1 of $0.487$ (default: $0.362$). Which variant is preferable therefore depends on the cost of a missed significant earthquake versus a false alarm; for hazard screening, where missing a $M \ge 4$ event is more costly, the balanced models are more appropriate even though their accuracy is lower.
3. **A compact, fully visible tree (Section 5.4, Fig. 23).** A depth-3 tree reaches $70.0\%$ test accuracy but only $0.463$ macro F1, and its rules are almost entirely about the magnitude-scale metadata: `type_Mw` and `type_Other` (upper-case `MW` labels) send events to `Minor` or `Light+`, and only then do `log_rms`, `depth` and `latitude` refine the decision. It illustrates the interpretability–accuracy trade-off: the depth-20 tree is $1.8$ points more accurate in accuracy and about $0.10$ higher in macro F1, but cannot be read as a whole.
4. **Dependence on catalogue metadata (Section 5.5, Fig. 26).** Removing the five `type_*` columns lowers the validation macro F1 of Gradient Boosting from $0.571$ to $0.435$ and the `Light+` recall from $0.269$ to $0.030$; additionally removing `rms`/`log_rms` changes little. Using only location and depth, `Light+` recall is $0.012$. In other words, **almost all of the ability to recognise `Light+` events comes from the magnitude-scale type reported in the catalogue**, which AFAD assigns mainly to larger events. The models should therefore be described as *estimating the severity class from catalogue metadata*, not as detecting significant earthquakes from location, depth and time alone. Temporal features also contribute (macro F1 $0.571 \rightarrow 0.542$ without them).
5. **Permutation importance (Section 5.6, Fig. 25).** On the validation set the largest drops in macro F1 are caused by shuffling `year` ($0.090$), `type_Other` ($0.080$), `type_Md` ($0.052$), `depth` ($0.047$) and `type_Mw` ($0.036$). The importance of `year` most likely reflects changes in the catalogue over time (denser station networks record more small events in recent years, and the magnitude type used changes over the years), rather than a physical trend in earthquake size. Gini importance (Fig. 15) ranks the continuous variables higher because impurity-based importance favours features with many possible split points.
6. **Uncertainty of the comparison (Section 5.7).** With only $960$ `Light+` test events, the 95% bootstrap interval of `Light+` recall is about $\pm 0.03$ (e.g. Gradient Boosting $0.303$ [$0.274$, $0.333$]). Even so, the paired bootstrap shows that Gradient Boosting is better than the *default* Random Forest on all four metrics (accuracy $+0.003$ [$+0.001$, $+0.005$], macro F1 $+0.022$ [$+0.013$, $+0.031$], `Light+` recall $+0.051$ [$+0.031$, $+0.073$]). The accuracy difference is statistically detectable but practically small.
7. **Province target encoding (Section 5.8).** Replacing the in-sample `province_seismic_index` with a 5-fold out-of-fold encoding changes test accuracy and weighted F1 by at most $0.002$ for both the Decision Tree and Gradient Boosting. The largest change is the Decision Tree `Light+` recall ($0.307 \rightarrow 0.279$), which is within its bootstrap interval. The in-sample encoding therefore does not materially affect the conclusions.

### 6.4. Conclusion
Gradient Boosting is the best classifier on the held-out testing set (accuracy $0.754$, weighted F1 $0.728$, macro F1 $0.584$), followed closely by Random Forest, with the Decision Tree offering the most interpretable rules and Logistic Regression clearly limited by its linear boundary. All models beat the majority-class baseline (macro F1 $0.267$) by a wide margin, and validation and testing results agree, so the 60/20/20 protocol gives reliable estimates. The main limitations are (i) the low recall on `Light+` events with default training, which cost-sensitive training can trade for precision, and (ii) the strong dependence on magnitude-scale metadata (`type_*`), which means the task is closer to *catalogue-based severity estimation* than to physical prediction.
